# Custodian: hosted Colab environment setup

This is the shared Issue #1 workflow. It verifies a reproducible hosted Google Colab environment and stops before dataset handling or model training. Do not connect to a local runtime, mount Google Drive, upload private files, capture traffic, replay packets, resolve observed domains, or place credentials in this notebook.


## 1. Required immutable revision and acknowledgement

Copy the complete reviewed 40-character commit SHA from GitHub. The exact acknowledgement is deliberately required. Software marker checks reject obvious local-runtime use but cannot prove isolation.


In [ ]:
CUSTODIAN_COMMIT = ""  # Required: full reviewed 40-character Git SHA.
HOSTED_COLAB_ACKNOWLEDGEMENT = ""  # Required exact phrase from docs/colab-training.md.

import os
import re
from pathlib import Path

EXPECTED_ACK = "I AM USING A HOSTED GOOGLE COLAB RUNTIME"
if HOSTED_COLAB_ACKNOWLEDGEMENT != EXPECTED_ACK:
    raise RuntimeError("Read docs/colab-training.md and provide the exact hosted-runtime acknowledgement.")
if not any(os.environ.get(name) for name in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
    raise RuntimeError("Hosted Colab markers are absent. Do not use Connect to local runtime.")
if os.environ.get("CUSTODIAN_ALLOW_LOCAL_RUNTIME"):
    raise RuntimeError("Local-runtime overrides are not supported.")
CUSTODIAN_COMMIT = CUSTODIAN_COMMIT.strip().lower()
if re.fullmatch(r"[0-9a-f]{40}", CUSTODIAN_COMMIT) is None:
    raise ValueError("CUSTODIAN_COMMIT must be a full 40-character Git commit SHA.")
CONTENT_ROOT = Path("/content")
if not CONTENT_ROOT.is_dir():
    raise RuntimeError("Expected hosted Colab /content directory is unavailable.")
print("Hosted-runtime guard passed; this is a guard, not proof of isolation.")


## 2. Create one owned workspace and checkout exactly that revision

The cell refuses to reuse or delete an existing directory. Use the explicit cleanup cell before starting over.


In [ ]:
import subprocess

WORKSPACE = CONTENT_ROOT / "custodian-workspace"
REPO = WORKSPACE / "repository"
OUTPUT = WORKSPACE / "output"
if WORKSPACE.exists():
    raise FileExistsError(f"Refusing to reuse {WORKSPACE}; inspect it or run the confirmed cleanup cell.")
OUTPUT.mkdir(parents=True)
subprocess.run(["git", "clone", "--no-checkout", "https://github.com/EmberFalls/Custodian.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", CUSTODIAN_COMMIT], cwd=REPO, check=True)
resolved_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip().lower()
if resolved_commit != CUSTODIAN_COMMIT:
    raise RuntimeError(f"Revision mismatch: expected {CUSTODIAN_COMMIT}, received {resolved_commit}")
print("Verified immutable Custodian revision:", resolved_commit)


## 3. Install exact direct pins without dependency mutation

Custodian is installed editable with `--no-deps` only after its direct Colab requirements are installed. The final resolved environment is recorded after installation.


In [ ]:
import sys

requirements = REPO / "training" / "requirements-colab.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Pinned Colab requirements missing at revision {resolved_commit}")
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(requirements)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(REPO)], check=True)
resolved_environment = subprocess.check_output([sys.executable, "-m", "pip", "freeze", "--all"], text=True).splitlines()
print("Recorded", len(resolved_environment), "resolved environment entries.")


## 4. Verify shared code and safety tests

This is a structural smoke test only. It does not train a model and produces no accuracy claim.


In [ ]:
os.chdir(REPO)
from training.colab import require_hosted_colab, require_owned_workspace, validate_commit_sha

validate_commit_sha(CUSTODIAN_COMMIT)
require_hosted_colab(HOSTED_COLAB_ACKNOWLEDGEMENT, environment=os.environ, content_root=CONTENT_ROOT)
require_owned_workspace(WORKSPACE, content_root=CONTENT_ROOT)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/unit/test_colab_workflow.py", "tests/unit/test_training_safety.py", "tests/unit/test_schemas.py", "tests/unit/test_features.py"], check=True)
print("Shared environment and feature-contract smoke tests passed. No model was trained.")


## 5. Export the environment manifest

The manifest contains no dataset rows, credentials, model artifacts, or claimed model results.


In [ ]:
import hashlib
import json
import platform
from datetime import UTC, datetime

manifest = {
    "manifest_version": "custodian.colab_environment.v1",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "repository_url": "https://github.com/EmberFalls/Custodian.git",
    "repository_commit": resolved_commit,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "hosted_colab_guard_passed": True,
    "local_runtime_supported": False,
    "dataset_processed": False,
    "model_trained": False,
    "resolved_environment": sorted(resolved_environment, key=str.casefold),
}
manifest_path = OUTPUT / "colab-environment-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
manifest_sha256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
print("Manifest:", manifest_path)
print("Manifest SHA-256:", manifest_sha256)
print(json.dumps({key: value for key, value in manifest.items() if key != "resolved_environment"}, indent=2))


In [ ]:
from google.colab import files as colab_files
colab_files.download(str(manifest_path))
print("After confirming the download, run the cleanup cell and delete the Colab runtime.")


## 6. Model-specific handoff

Stop here for Issue #1. A later model issue must review its dataset adapter, shared feature schema, grouped splits, calibration, evaluation, and artifact policy before it deliberately opens Custodian's isolated-training gate.


## 7. Explicit cleanup

Run only after the manifest is downloaded. This cell deletes exactly `/content/custodian-workspace`, then you must use **Runtime → Disconnect and delete runtime**.


In [ ]:
CLEANUP_CONFIRMATION = ""  # Required exact phrase: DELETE CUSTODIAN COLAB WORKSPACE

import shutil
from training.colab import require_owned_workspace

if CLEANUP_CONFIRMATION != "DELETE CUSTODIAN COLAB WORKSPACE":
    raise RuntimeError("Cleanup confirmation missing; no files were removed.")
owned_workspace = require_owned_workspace(WORKSPACE, content_root=CONTENT_ROOT)
if owned_workspace.exists():
    shutil.rmtree(owned_workspace)
print("Removed the dedicated Custodian workspace. Now disconnect and delete the Colab runtime.")
